# セットアップ
Google Driveをマウントし、rcloneで同期済みのプロジェクトフォルダに移動する。
データ(images・labeled_data等)はすべて事前にローカルPC側で生成・rclone同期済みであることが前提。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/datafetch_modeltrain

# 💡 train.py / evaluate.py はPyTorch標準のみで動作するため追加インストール不要。
# stable-baselines3 / gymnasium / mplfinance は run_colab_pipeline.py 側でも
# 自動インストールされるが、ここで明示的に入れておく。
!pip install -q "stable-baselines3==2.9.0" gymnasium mplfinance

Mounted at /content/drive
/content/drive/MyDrive/datafetch_modeltrain


# 設定
以後のセルはすべてここの変数を参照する。モードや期間を変える場合はここだけ編集すればよい。

⚠️ `MODE`は`run_colab_pipeline.py`冒頭の`MODE = "medium"`と一致させておくこと(別ファイルのため自動連動しない)。

💡 `FREEZE_BACKBONE`(2026-11-13追加): train.pyのResNet50バックボーン凍結設定。選択肢と精度への影響の考え方はREADME.md参照。

In [ ]:
MODE = "medium"
SYMBOLS = ["GBPUSD", "EURUSD", "USDJPY", "AUDUSD", "GBPJPY", "EURJPY", "GOLD"]

# split_data.py: ポートフォリオRL用データの学習開始日(testは自動算出される70/20/10分割)
TRAIN_START_DATE = "2022-08-01"

# train.py: 分析モデル(教師あり学習)のエポック数
ANALYST_EPOCHS = 10

# train.py: ResNet50バックボーンの凍結範囲。"none"/"stem"/"early"/"early_plus_layer3"
# (既存モデルをそのまま継続する場合は"none"のままでよい。詳細はREADME.md参照)
FREEZE_BACKBONE = "none"

print(f"MODE={MODE} / SYMBOLS={SYMBOLS} / TRAIN_START_DATE={TRAIN_START_DATE} / ANALYST_EPOCHS={ANALYST_EPOCHS} / FREEZE_BACKBONE={FREEZE_BACKBONE}")

# 事前チェック
必要なデータ(ラベル付き・スケール済みCSV、チャート画像)がすべて同期済みか確認する。
データ取得(`fetch_data.py`)・スケーラー作成(`apply_scaler.py`等、`run_pipeline.py`が担当)はColabでは実行しない(MT5必須・Windows専用のため)。ここではそれらの成果物が揃っているかだけを確認する。

In [ ]:
import os

missing = []
for sym in SYMBOLS:
    for suffix in ["scaled", "labeled"]:
        p = f"labeled_data/{sym}_{MODE}_{suffix}.csv"
        if not os.path.exists(p):
            missing.append(p)
    img_dir = f"images/{sym}/{MODE}_MTF"
    if not os.path.isdir(img_dir) or not any(f.endswith(".png") for f in os.listdir(img_dir)):
        missing.append(img_dir)

if missing:
    print("❌ 以下が見つかりません。ローカル側での生成・rcloneでの同期状況を確認してください:")
    for m in missing:
        print(f"  - {m}")
else:
    print("✅ 必要なデータが揃っています。学習を開始できます。")

OSError: [Errno 5] Input/output error: 'images/GBPUSD/medium_MTF'

# 1. 教師あり学習(分析モデル: MultimodalFXmodel)
`train.py`が複数の評価基準(Total/Class/Risk/最終epoch)でモデルを保存し、`evaluate.py`がその中から実際のテストデータで最も精度の高いものを選定する。

💡 2026-11-13の改修: 既存モデルの読み込みに失敗した場合、学習は停止するようになった(以前は静かに新規学習にフォールバックしていた)。`FREEZE_BACKBONE`でResNet50バックボーンの凍結範囲も選べる(詳細はREADME.md参照)。

In [ ]:
!python train.py --mode {MODE} --epochs {ANALYST_EPOCHS} --freeze-backbone {FREEZE_BACKBONE}

In [ ]:
!python evaluate.py --mode {MODE}

`evaluate.py`は選定したモデルを`/content/drive/MyDrive/FX_AI_Models/final_best_model_{mode}.pth`に保存する(このノートブックの作業ディレクトリとは別の固定パス)。
`precompute_portfolio.py`はカレントディレクトリ相対の`saved_model/final_best_model_{mode}.pth`を探すため、ここで正しい場所にコピーしておく。

In [ ]:
import shutil

drive_best_model = f"/content/drive/MyDrive/FX_AI_Models/final_best_model_{MODE}.pth"
local_best_model = f"saved_model/final_best_model_{MODE}.pth"

if os.path.exists(drive_best_model):
    os.makedirs("saved_model", exist_ok=True)
    shutil.copy(drive_best_model, local_best_model)
    print(f"✅ コピー完了: {local_best_model}")
else:
    print(f"❌ {drive_best_model} が見つかりません。evaluate.pyの実行結果を確認してください。")

# 2. 強化学習用データの準備(銘柄別に学習・テスト期間を分割)
`merge_portfolio_data.py`が必要とする`_train`/`_test`付きファイルをここで作成する。

In [ ]:
for sym in SYMBOLS:
    !python split_data.py --symbol {sym} --mode {MODE} --train_start_date {TRAIN_START_DATE}

2026-10-30 15:48:05,966 [INFO] split_data.py ver209.0
2026-10-30 15:48:16,092 [INFO] 📐 --split_date 未指定のため、train.pyと同じ70/20/10比率から境界日を自動計算しました: 2026-04-02 09:00:00
2026-10-30 15:48:16,094 [INFO] データをロード中... (学習開始日:2022-08-01,境界日: 2026-04-02 09:00:00)
2026-10-30 15:49:20,837 [INFO] ✅ データの分割と保存が完了しました！
2026-10-30 15:49:20,838 [INFO] 📊 Trainデータ件数: 90801 行 -> labeled_data/GBPUSD_medium_scaled_train.csv
2026-10-30 15:49:20,838 [INFO] 📊 Testデータ件数:  10089 行 -> labeled_data/GBPUSD_medium_scaled_test.csv
2026-10-30 15:49:22,001 [INFO] split_data.py ver209.0
2026-10-30 15:49:37,305 [INFO] 📐 --split_date 未指定のため、train.pyと同じ70/20/10比率から境界日を自動計算しました: 2026-04-02 08:45:00
2026-10-30 15:49:37,307 [INFO] データをロード中... (学習開始日:2022-08-01,境界日: 2026-04-02 08:45:00)
2026-10-30 15:50:39,515 [INFO] ✅ データの分割と保存が完了しました！
2026-10-30 15:50:39,515 [INFO] 📊 Trainデータ件数: 90801 行 -> labeled_data/EURUSD_medium_scaled_train.csv
2026-10-30 15:50:39,515 [INFO] 📊 Testデータ件数:  10090 行 -> labeled_data/EURUSD_medium_scaled_test.cs

# 3. 強化学習(ポートフォリオSACエージェント)
`run_colab_pipeline.py`が以下を順に実行する:
- 既存モデルの退避
- Google Driveからの既存データ復元 + 既に揃っていればデータ処理をスキップ
- `merge_portfolio_data.py`(train/test統合)
- `precompute_portfolio.py`(train/test、分析モデルの推論結果を事前付与)
- `train_portfolio_rl.py`(SACエージェントの学習)

下のTensorBoardセルは、学習セルを実行する前にこのセルだけ先に実行しておくと、学習の進捗をリアルタイムに確認できる。

In [ ]:
%load_ext tensorboard
%tensorboard --logdir ./sac_portfolio_tensorboard

In [ ]:
!python run_colab_pipeline.py

 🌍 Google Colab ポートフォリオ学習パイプライン
run_colab_pipeline.py ver210.0

📦 パッケージをインストールしています...
🚀 実行中: pip install stable-baselines3==2.9.0 gymnasium mplfinance
✅ 完了: pip install stable-baselines3==2.9.0 gymnasium mplfinance
----------------------------------------
📁 必須ディレクトリを確認・作成しました。

=== 既存のポートフォリオRLモデルを退避 ===
  (退避対象の既存モデル・チェックポイントは見つかりませんでした。新規学習として開始します)

=== Google Driveから既存データの復元を試みます ===

✅ 事前推論済みの学習・テストデータ(portfolio_merged_data_{train,test}_with_preds.csv)が既に揃っているため、データの結合・事前推論処理をスキップします。
Traceback (most recent call last):
  File "/content/drive/MyDrive/datafetch_modeltrain/run_colab_pipeline.py", line 204, in <module>
    main()
    ~~~~^^
  File "/content/drive/MyDrive/datafetch_modeltrain/run_colab_pipeline.py", line 193, in main
    warn_if_images_missing()
    ~~~~~~~~~~~~~~~~~~~~~~^^
  File "/content/drive/MyDrive/datafetch_modeltrain/run_colab_pipeline.py", line 128, in warn_if_images_missing
    if not os.path.isdir(img_dir) or not any(f.endswith(".png") for f in os.listdir(i

# 4. テスト(バックテスト・性能評価)
学習済みモデルで静的なテストデータを1回走らせ、勝率・プロフィットファクター・月利換算・最大ドローダウン等を集計する(MT5への接続は行わないため、ヒストリカルデータの再現性問題の影響を受けない)。

In [ ]:
!python test_portfolio.py --mode {MODE}

In [ ]:
!python test_portfolio.py --mode medium --entry-threshold-window 300 --entry-threshold-percentile 60 --max-concurrent-positions 5 --risk-per-trade-pct 7 --n-splits 3

!python test_portfolio.py --mode medium --entry-threshold-window 500 --entry-threshold-percentile 50   --max-concurrent-positions 5 --risk-per-trade-pct 7 --n-splits 3

ストリーミング出力は最後の 5000 行に切り捨てられました。
2026-11-06 07:12:34,829 [INFO] Step 0330 | Equity: 10,265,701 円 | Balance: 9,960,681 円
2026-11-06 07:12:34,829 [INFO]     [GBPUSD] pos=+1.43lot 価格=1.325 含み損益=76,085円 直近の実現損益=0円
2026-11-06 07:12:34,830 [INFO]     [EURUSD] pos=+1.62lot 価格=1.156 含み損益=61,604円 直近の実現損益=0円
2026-11-06 07:12:34,830 [INFO]     [USDJPY] pos=-1.63lot 価格=159.778 含み損益=-39,772円 直近の実現損益=0円
2026-11-06 07:12:34,830 [INFO]     [GBPJPY] pos=+1.36lot 価格=211.684 含み損益=88,128円 直近の実現損益=0円
2026-11-06 07:12:34,830 [INFO]     [EURJPY] pos=+1.69lot 価格=184.737 含み損益=118,976円 直近の実現損益=0円
2026-11-06 07:12:34,850 [INFO] Step 0340 | Equity: 10,262,390 円 | Balance: 9,960,681 円
2026-11-06 07:12:34,850 [INFO]     [GBPUSD] pos=+1.43lot 価格=1.324 含み損益=61,051円 直近の実現損益=0円
2026-11-06 07:12:34,850 [INFO]     [EURUSD] pos=+1.62lot 価格=1.156 含み損益=61,910円 直近の実現損益=0円
2026-11-06 07:12:34,850 [INFO]     [USDJPY] pos=-1.63lot 価格=159.899 含み損益=-59,495円 直近の実現損益=0円
2026-11-06 07:12:34,850 [INFO]     [GBPJPY] pos=+1.36lot 価格=211

In [ ]:
!python test_portfolio.py --mode medium --entry-threshold-window 500 --entry-threshold-percentile 40 --max-concurrent-positions 5 --risk-per-trade-pct 7 --n-splits 3

I0000 00:00:1793951076.410903   53211 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793951076.482210   53211 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793951136.095325   53211 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793951136.095881   53211 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.

# 5. (任意)動的エントリー閾値のパラメータ探索
固定の発注閾値(0.2)を、銘柄ごとの直近の行動分布に基づく動的な閾値に置き換えるための検証。モデルの再学習は行わず、既存の学習済みモデルをそのまま使って複数パターンをバックテストし、月利換算・最大ドローダウンの目標を満たす設定を探す。

In [ ]:
!python sweep_entry_threshold.py --mode {MODE}

I0000 00:00:1793853826.715992    6562 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793853826.784515    6562 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793853838.303861    6562 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793853838.304467    6562 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.

In [ ]:
!python sweep_entry_threshold.py --mode {MODE} --windows 300 --percentiles 50,60 --max-concurrent-positions 5 --risk-per-trade-pct 7,8,9,10,11,12,13,14,15

I0000 00:00:1793866866.739615    3057 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793866866.806616    3057 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793866876.957585    3057 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793866876.958172    3057 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.

In [ ]:
!python sweep_entry_threshold.py --mode medium --windows 100,200,300,500 --percentiles 40,50,60,70,80 --max-concurrent-positions 3,4,5,6,7 --risk-per-trade-pct 7,8

I0000 00:00:1793893268.304923    3376 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793893268.376683    3376 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793893280.975142    3376 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793893280.975773    3376 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.

In [ ]:
!python test_portfolio.py --mode medium --entry-threshold-window 200 --entry-threshold-percentile 70 --max-concurrent-positions 5 --risk-per-trade-pct 7 --n-splits 3

ストリーミング出力は最後の 5000 行に切り捨てられました。
2026-11-06 06:19:32,702 [INFO]     [GOLD] pos=-0.09lot 価格=4806.680 含み損益=-52,311円 直近の実現損益=0円
2026-11-06 06:19:32,729 [INFO] Step 0400 | Equity: 10,965,028 円 | Balance: 10,735,277 円
2026-11-06 06:19:32,730 [INFO]     [GBPUSD] pos=-1.41lot 価格=1.342 含み損益=-144,472円 直近の実現損益=0円
2026-11-06 06:19:32,730 [INFO]     [USDJPY] pos=-1.63lot 価格=158.365 含み損益=190,547円 直近の実現損益=0円
2026-11-06 06:19:32,730 [INFO]     [GBPJPY] pos=+1.36lot 価格=212.477 含み損益=195,976円 直近の実現損益=0円
2026-11-06 06:19:32,730 [INFO]     [EURJPY] pos=-1.23lot 価格=184.932 含み損益=43,542円 直近の実現損益=0円
2026-11-06 06:19:32,730 [INFO]     [GOLD] pos=-0.09lot 価格=4809.190 含み損益=-55,843円 直近の実現損益=0円
2026-11-06 06:19:32,758 [INFO] Step 0410 | Equity: 10,929,797 円 | Balance: 10,760,126 円
2026-11-06 06:19:32,758 [INFO]     [GBPJPY] pos=+1.36lot 価格=212.391 含み損益=184,280円 直近の実現損益=0円
2026-11-06 06:19:32,758 [INFO]     [EURJPY] pos=-1.23lot 価格=184.956 含み損益=40,590円 直近の実現損益=0円
2026-11-06 06:19:32,758 [INFO]     [GOLD] pos=-0.09lo

#画像の欠損分布


In [ ]:
!python diagnose_missing_images.py --phase train --mode medium